# Sesión 2: Transformaciones básicas de datos en Power Query
## Códigos M - Ejemplos y Ejercicios

Este notebook contiene el código M de la presentación **Transformaciones básicas de datos en Power Query** para copiar y pegar directamente en el Editor de Power Query de Power BI.

---
## SLIDE 6: Quitar columnas y filas innecesarias

### 1. Quitar columnas específicas

```m
let
    origen = Excel.Workbook(
        File.Contents("datos_clientes.xlsx"),
        null,
        true
    ),
    tabla = origen{0}[Data],
    con_encabezados = Table.PromoteHeaders(tabla),
    
    // Eliminar columnas que no aportan valor al análisis
    sin_columnas_innecesarias = Table.RemoveColumns(
        con_encabezados,
        {"ColumnaTemporal", "NoUsada", "LegadoSistema", "FechaCarga"}
    )
in
    sin_columnas_innecesarias
```

**Explicación:**
- `Table.RemoveColumns()`: Elimina las columnas especificadas
- Reduce el volumen de datos mejorando rendimiento
- Documentar por qué se elimina cada columna

**Buenas prácticas:**
- Verificar si la columna afecta relaciones futuras
- Considerar implicancias legales o regulatorias
- Usar nombres claros en los pasos

---

### 2. Quitar filas vacías

```m
let
    origen = Excel.Workbook(
        File.Contents("ventas.xlsx"),
        null,
        true
    ),
    tabla = origen{0}[Data],
    con_encabezados = Table.PromoteHeaders(tabla),
    
    // Eliminar filas completamente vacías
    sin_filas_vacias = Table.SelectRows(
        con_encabezados,
        each not List.IsEmpty(
            List.RemoveMatchingItems(
                Record.FieldValues(_),
                {null, ""}
            )
        )
    )
in
    sin_filas_vacias
```

**Explicación:**
- `Table.SelectRows()`: Filtra filas según condición
- `List.IsEmpty()`: Verifica si la lista está vacía
- `Record.FieldValues()`: Extrae todos los valores de una fila
- Detecta espacios o caracteres invisibles

---

### 3. Quitar filas duplicadas

```m
let
    origen = Csv.Document(
        File.Contents("registros.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // Eliminar duplicados basados en todas las columnas
    sin_duplicados_completos = Table.Distinct(
        con_encabezados,
        Comparer.OrdinalIgnoreCase
    )
in
    sin_duplicados_completos
```

**Nota:** Si necesitas eliminar duplicados basados en columnas específicas (ej: ClienteID):

```m
let
    origen = Csv.Document(
        File.Contents("registros.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // Eliminar duplicados por ClienteID (mantiene el primero)
    sin_duplicados_por_clave = Table.Distinct(
        con_encabezados,
        {"ClienteID"},
        Comparer.OrdinalIgnoreCase
    )
in
    sin_duplicados_por_clave
```

**Explicación:**
- `Table.Distinct()`: Elimina filas duplicadas
- Sin parámetros: compara todas las columnas
- Con columnas especificadas: solo compara esas claves
- `Comparer.OrdinalIgnoreCase`: No distingue mayúsculas/minúsculas

---

### 4. Quitar filas con errores

```m
let
    origen = Csv.Document(
        File.Contents("datos_mixtos.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"Monto", type number}
    ),
    
    // Eliminar filas donde la conversión de tipo falló
    sin_errores = Table.SelectRows(
        cambiar_tipos,
        each not [Monto] = null and not [Monto] = error
    )
in
    sin_errores
```

**Alternativa más directa:**

```m
let
    origen = Csv.Document(
        File.Contents("datos_mixtos.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // En Power Query, usar filtro visual para mostrar solo errores
    // luego aplicar este paso para excluir errores
    sin_errores = Table.RemoveRowsWithErrors(
        con_encabezados
    )
in
    sin_errores
```

---

## SLIDE 7: Dividir columnas

### 1. Dividir por delimitador

```m
let
    origen = Csv.Document(
        File.Contents("ubicaciones.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // Dividir "Region - Ciudad" por guion
    dividir_por_guion = Table.SplitColumn(
        con_encabezados,
        "Ubicacion",
        Splitter.SplitTextByDelimiter("-", QuoteStyle.Csv),
        {"Region", "Ciudad"}
    ),
    
    // Limpiar espacios en las columnas resultantes
    limpiar_espacios = Table.TransformColumns(
        dividir_por_guion,
        {
            {"Region", Text.Trim, type text},
            {"Ciudad", Text.Trim, type text}
        }
    )
in
    limpiar_espacios
```

**Explicación:**
- `Table.SplitColumn()`: Divide una columna
- `Splitter.SplitTextByDelimiter()`: Define el delimitador
- Se especifican los nombres de las nuevas columnas
- `Text.Trim`: Elimina espacios adicionales

**Delimitadores comunes:**
- `"-"`: Guion
- `","`: Coma
- `" "`: Espacio
- `"/"`: Barra
- `"_"`: Guion bajo

---

### 2. Dividir por número fijo de caracteres

```m
let
    origen = Excel.Workbook(
        File.Contents("codigos.xlsx"),
        null,
        true
    ),
    tabla = origen{0}[Data],
    con_encabezados = Table.PromoteHeaders(tabla),
    
    // Dividir código de 8 caracteres: AAA-NNNNN
    // Primeros 3 caracteres = categoría, resto = secuencial
    dividir_por_longitud = Table.SplitColumn(
        con_encabezados,
        "Codigo",
        Splitter.SplitTextByLengths({3, 5}),
        {"Categoria", "Secuencial"}
    )
in
    dividir_por_longitud
```

**Explicación:**
- `Splitter.SplitTextByLengths({3, 5})`: Divide en fragmentos de 3 y 5 caracteres
- Útil para códigos con estructura fija
- {3, 5} significa: toma 3 caracteres, luego 5, luego el resto

---

### 3. Dividir hasta encontrar delimitador (usando expresión regular - avanzado)

```m
let
    origen = Csv.Document(
        File.Contents("nombres.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // Separar apellido y nombre (asume: "Apellido, Nombre")
    dividir_nombre = Table.SplitColumn(
        con_encabezados,
        "NombreCompleto",
        Splitter.SplitTextByDelimiter(",", QuoteStyle.Csv),
        {"Apellido", "Nombre"}
    ),
    
    // Limpiar espacios
    limpiar_nombres = Table.TransformColumns(
        dividir_nombre,
        {
            {"Apellido", Text.Trim, type text},
            {"Nombre", Text.Trim, type text}
        }
    )
in
    limpiar_nombres
```

---

## SLIDE 8: Agrupar datos

### 1. Group By básico - Contar por grupo

```m
let
    origen = Csv.Document(
        File.Contents("solicitudes.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"Fecha", type date}
    ),
    
    // Agrupar por tipo de solicitud y contar
    agrupar_por_tipo = Table.Group(
        cambiar_tipos,
        {"TipoSolicitud"},
        {{"CantidadSolicitudes", Table.RowCount, Int64.Type}}
    )
in
    agrupar_por_tipo
```

**Explicación:**
- `Table.Group()`: Agrupa por una o más columnas
- `{"TipoSolicitud"}`: Columna por la que agrupar
- `Table.RowCount`: Cuenta filas en cada grupo
- Resultado: tabla con tipo único y cantidad

---

### 2. Múltiples agregaciones simultáneas

```m
let
    origen = Csv.Document(
        File.Contents("ventas.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"Monto", type number, "Fecha", type date}
    ),
    
    // Agrupar por Agente con múltiples agregaciones
    agrupar_multiples = Table.Group(
        cambiar_tipos,
        {"Agente"},
        {
            {"TotalVentas", each List.Sum([Monto]), type number},
            {"VentaPromedio", each List.Average([Monto]), type number},
            {"MaximoVenta", each List.Max([Monto]), type number},
            {"MinimoVenta", each List.Min([Monto]), type number},
            {"QuantidadTransacciones", Table.RowCount, Int64.Type}
        }
    )
in
    agrupar_multiples
```

**Explicación:**
- Define múltiples funciones de agregación en un solo Group
- `List.Sum()`: Suma
- `List.Average()`: Promedio
- `List.Max()`: Máximo
- `List.Min()`: Mínimo
- `Table.RowCount`: Conteo

---

### 3. Agrupación por múltiples columnas

```m
let
    origen = Csv.Document(
        File.Contents("ventas_detalle.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"Monto", type number, "Fecha", type date}
    ),
    
    // Agregar columnas de Año y Mes
    agregar_año_mes = Table.AddColumn(
        cambiar_tipos,
        "Año",
        each Date.Year([Fecha]),
        type number
    ),
    agregar_mes = Table.AddColumn(
        agregar_año_mes,
        "Mes",
        each Date.Month([Fecha]),
        type number
    ),
    
    // Agrupar por Región, Año y Mes
    agrupar_multiples = Table.Group(
        agregar_mes,
        {"Region", "Año", "Mes"},
        {
            {"TotalMensual", each List.Sum([Monto]), type number},
            {"PromedioDiario", each List.Average([Monto]), type number}
        }
    )
in
    agrupar_multiples
```

**Explicación:**
- Agrupa por Región, Año y Mes
- Calcula total mensual y promedio
- Resultado: resumen por cada combinación de región-año-mes

---

## SLIDE 9: Columnas calculadas simples

### 1. Columna condicional básica

```m
let
    origen = Csv.Document(
        File.Contents("ventas.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"Monto", type number}
    ),
    
    // Crear columna que clasifique por meta cumplida
    agregar_clasificacion = Table.AddColumn(
        cambiar_tipos,
        "CumpleMeta",
        each if [Monto] >= 500000 then "Cumple" else "No cumple",
        type text
    )
in
    agregar_clasificacion
```

**Explicación:**
- `Table.AddColumn()`: Agrega una nueva columna
- `each`: Aplica la lógica a cada fila
- `if...then...else`: Estructura condicional
- Tipos posibles: text, number, logical, date

---

### 2. Concatenación de columnas

```m
let
    origen = Excel.Workbook(
        File.Contents("clientes.xlsx"),
        null,
        true
    ),
    tabla = origen{0}[Data],
    con_encabezados = Table.PromoteHeaders(tabla),
    
    // Concatenar Nombre y Apellido
    agregar_nombre_completo = Table.AddColumn(
        con_encabezados,
        "NombreCompleto",
        each [Nombre] & " " & [Apellido],
        type text
    )
in
    agregar_nombre_completo
```

**Explicación:**
- `&`: Operador de concatenación
- Combina múltiples campos con separadores
- Útil para crear códigos o identificadores compuestos

---

### 3. Operación matemática

```m
let
    origen = Csv.Document(
        File.Contents("facturas.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"MontoBase", type number, "Descuento", type number}
    ),
    
    // Calcular monto final después de descuento
    agregar_monto_final = Table.AddColumn(
        cambiar_tipos,
        "MontoFinal",
        each [MontoBase] * (1 - [Descuento] / 100),
        type number
    )
in
    agregar_monto_final
```

**Explicación:**
- Operaciones: +, -, *, /, mod (módulo)
- Aplica lógica matemática a cada fila
- Resulta en nueva columna numérica

---

### 4. Diferencia entre fechas

```m
let
    origen = Csv.Document(
        File.Contents("tareas.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"FechaInicio", type date, "FechaFin", type date}
    ),
    
    // Calcular duración en días
    agregar_dias_duracion = Table.AddColumn(
        cambiar_tipos,
        "DiasDuracion",
        each Duration.Days([FechaFin] - [FechaInicio]),
        type number
    )
in
    agregar_dias_duracion
```

**Explicación:**
- Resta fechas para obtener duración
- `Duration.Days()`: Convierte a número de días
- Otras opciones: `Duration.Months()`, `Duration.Years()`, `Duration.Hours()`

---

## SLIDE 13-14: Tipos de datos disponibles

### Referencia: Conversión de tipos de datos

```m
let
    origen = Csv.Document(
        File.Contents("datos_mixtos.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // Conversión de tipos
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {
            {"ID", Int64.Type},                    // Número entero
            {"Nombre", type text},                 // Texto
            {"Precio", Decimal.Type},              // Número decimal
            {"Stock", Int64.Type},                 // Número entero
            {"EsActivo", type logical},            // Booleano (verdadero/falso)
            {"FechaCreacion", type date},          // Fecha
            {"HoraRegistro", type time},           // Hora
            {"FechaHoraActualizacion", type datetime}  // Fecha y hora
        }
    )
in
    cambiar_tipos
```

**Tipos disponibles:**
- `type text`: Texto/cadena
- `Int64.Type`: Número entero (64 bits)
- `type number`: Número decimal automático
- `Decimal.Type`: Número decimal de precisión fija
- `type date`: Fecha
- `type time`: Hora
- `type datetime`: Fecha y hora
- `type logical`: Booleano (true/false)
- `type duration`: Duración

---

### Detección automática de tipo de dato (Slide 15)

```m
let
    origen = Excel.Workbook(
        File.Contents("datos.xlsx"),
        null,
        true
    ),
    tabla = origen{0}[Data],
    con_encabezados = Table.PromoteHeaders(tabla),
    
    // Power Query intenta detectar tipos automáticamente
    // Pero es recomendable especificar explícitamente
    tipos_detectados = Table.TransformColumnTypes(
        con_encabezados,
        List.Transform(
            Table.ColumnNames(con_encabezados),
            each {_, type text}  // Inicialmente todo como texto
        )
    ),
    
    // Luego cambiar específicamente los que sabemos
    tipos_correctos = Table.TransformColumnTypes(
        tipos_detectados,
        {{"Fecha", type date}, {"Monto", type number}}
    )
in
    tipos_correctos
```

---

## SLIDE 20: Reemplazo de valores

### 1. Reemplazo directo (simple)

```m
let
    origen = Csv.Document(
        File.Contents("solicitudes.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // Reemplazar "Sin información" por "No informado"
    reemplazar_simple = Table.ReplaceValue(
        con_encabezados,
        "Sin información",
        "No informado",
        Replacer.ReplaceText,
        {"Estado"}  // Solo en columna "Estado"
    )
in
    reemplazar_simple
```

**Explicación:**
- `Table.ReplaceValue()`: Reemplaza valores
- `Replacer.ReplaceText`: Busca texto exacto
- `{"Estado"}`: Limita a columna específica
- Usa `null` para el segundo parámetro si busca en todas las columnas

---

### 2. Reemplazo condicional con lógica if...then...else

```m
let
    origen = Csv.Document(
        File.Contents("encuestas.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"Edad", Int64.Type}
    ),
    
    // Reemplazar "S/I" por "No informado" solo si Edad < 18
    reemplazar_condicional = Table.AddColumn(
        cambiar_tipos,
        "GeneroNormalizado",
        each if [Edad] < 18 and [Genero] = "S/I" then
                "No informado (menor)"
            else if [Genero] = "S/I" then
                "No informado"
            else
                [Genero],
        type text
    ),
    
    // Opcionalmente, eliminar columna original
    sin_genero_original = Table.RemoveColumns(
        reemplazar_condicional,
        {"Genero"}
    ),
    
    // Renombrar la nueva columna
    renombrar = Table.RenameColumns(
        sin_genero_original,
        {{"GeneroNormalizado", "Genero"}}
    )
in
    renombrar
```

**Explicación:**
- Estructura `if...then...else` anidada
- Reemplaza basado en múltiples condiciones
- Crea nueva columna con valores corregidos

---

### 3. Reemplazo masivo (múltiples valores)

```m
let
    origen = Csv.Document(
        File.Contents("categorias.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // Múltiples reemplazos secuenciales
    reemplazar_1 = Table.ReplaceValue(
        con_encabezados,
        "F",
        "Femenino",
        Replacer.ReplaceText,
        {"Genero"}
    ),
    reemplazar_2 = Table.ReplaceValue(
        reemplazar_1,
        "M",
        "Masculino",
        Replacer.ReplaceText,
        {"Genero"}
    ),
    reemplazar_3 = Table.ReplaceValue(
        reemplazar_2,
        "O",
        "Otro",
        Replacer.ReplaceText,
        {"Genero"}
    )
in
    reemplazar_3
```

**Mejor práctica: usar tabla de mapeo**

```m
let
    origen = Csv.Document(
        File.Contents("categorias.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // Tabla de equivalencias
    tabla_mapeo = #table(
        {"Valor_Antiguo", "Valor_Nuevo"},
        {
            {"F", "Femenino"},
            {"M", "Masculino"},
            {"O", "Otro"},
            {"S/I", "No informado"}
        }
    ),
    
    // Merge con tabla de mapeo para reemplazo masivo
    con_mapeo = Table.NestedJoin(
        con_encabezados,
        {"Genero"},
        tabla_mapeo,
        {"Valor_Antiguo"},
        "Mapeo"
    ),
    
    expandir_mapeo = Table.ExpandTableColumn(
        con_mapeo,
        "Mapeo",
        {"Valor_Nuevo"},
        {"Genero_Nuevo"}
    ),
    
    usar_nuevo = Table.AddColumn(
        expandir_mapeo,
        "Genero_Final",
        each if [Genero_Nuevo] <> null then [Genero_Nuevo] else [Genero],
        type text
    ),
    
    limpiar = Table.RemoveColumns(
        usar_nuevo,
        {"Genero", "Genero_Nuevo"}
    ),
    
    renombrar = Table.RenameColumns(
        limpiar,
        {{"Genero_Final", "Genero"}}
    )
in
    renombrar
```

**Ventajas de tabla de mapeo:**
- Centraliza todos los reemplazos
- Fácil de actualizar
- Reutilizable en múltiples consultas
- Mejora trazabilidad

---

### 4. Reemplazo de valores nulos

```m
let
    origen = Excel.Workbook(
        File.Contents("datos_incompletos.xlsx"),
        null,
        true
    ),
    tabla = origen{0}[Data],
    con_encabezados = Table.PromoteHeaders(tabla),
    
    // Reemplazar valores nulos en columna específica
    reemplazar_nulos = Table.ReplaceValue(
        con_encabezados,
        null,
        "Sin dato",
        Replacer.ReplaceValue,
        {"Observaciones"}
    )
in
    reemplazar_nulos
```

---

### 5. Reemplazo con expresiones regulares (avanzado)

```m
let
    origen = Csv.Document(
        File.Contents("telefonos.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // Normalizar teléfono: eliminar guiones, espacios, paréntesis
    normalizar_telefono = Table.TransformColumns(
        con_encabezados,
        {
            "Telefono",
            each Text.Replace(
                Text.Replace(
                    Text.Replace(
                        Text.Replace([Telefono], "-", ""),
                        " ", ""
                    ),
                    "(", ""
                ),
                ")", ""
            ),
            type text
        }
    )
in
    normalizar_telefono
```

---

## SLIDE 27: Tipos de filtrado en Power Query

### 1. Filtrado por valor específico

```m
let
    origen = Csv.Document(
        File.Contents("ordenes.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // Mantener solo estado "Completado"
    filtrar_completados = Table.SelectRows(
        con_encabezados,
        each [Estado] = "Completado"
    )
in
    filtrar_completados
```

**Explicación:**
- `Table.SelectRows()`: Filtra filas
- `each [Estado] = "Completado"`: Condición
- Solo conserva filas que cumplen

---

### 2. Filtrado por condición (mayor que, contiene, etc.)

```m
let
    origen = Csv.Document(
        File.Contents("transacciones.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"Monto", type number}
    ),
    
    // Mantener solo transacciones mayores a $1.000.000
    filtrar_mayores = Table.SelectRows(
        cambiar_tipos,
        each [Monto] > 1000000
    )
in
    filtrar_mayores
```

**Otros operadores:**
- `<`: Menor que
- `>=`: Mayor o igual
- `<=`: Menor o igual
- `<>`: Diferente
- `Text.Contains()`: Contiene texto
- `Text.StartsWith()`: Empieza con
- `Text.EndsWith()`: Termina con

---

### 3. Filtrado de fechas

```m
let
    origen = Csv.Document(
        File.Contents("ventas.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"Fecha", type date}
    ),
    
    // Mantener solo del año 2024
    filtrar_2024 = Table.SelectRows(
        cambiar_tipos,
        each Date.Year([Fecha]) = 2024
    )
in
    filtrar_2024
```

**Ejemplos adicionales:**

```m
// Últimos 30 días
each [Fecha] >= Date.AddDays(Date.From(DateTime.LocalNow()), -30)

// Mes actual
each Date.Year([Fecha]) = Date.Year(Date.From(DateTime.LocalNow())) and
     Date.Month([Fecha]) = Date.Month(Date.From(DateTime.LocalNow()))

// Entre dos fechas
each [Fecha] >= #date(2024, 1, 1) and [Fecha] <= #date(2024, 12, 31)
```

---

### 4. Filtrado dinámico con parámetros (Slide 29)

```m
// Primero, crear un PARÁMETRO llamado "FechaLimite"
// Clic derecho en "Parámetros" → Nuevo parámetro
// Nombre: FechaLimite
// Tipo: Date
// Valor predeterminado: #date(2024, 1, 1)

let
    origen = Csv.Document(
        File.Contents("transacciones.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"Fecha", type date}
    ),
    
    // Usar el parámetro para filtrar dinámicamente
    filtrar_dinamico = Table.SelectRows(
        cambiar_tipos,
        each [Fecha] >= FechaLimite  // FechaLimite es el parámetro
    )
in
    filtrar_dinamico
```

**Ventajas:**
- Cambiar fecha límite sin editar la consulta
- Reutilizar para diferentes períodos
- Facilita automatización

---

### 5. Filtrado de errores o valores nulos

```m
let
    origen = Csv.Document(
        File.Contents("datos_mixtos.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    
    // Eliminar filas donde Teléfono es nulo o vacío
    sin_nulos = Table.SelectRows(
        con_encabezados,
        each [Telefono] <> null and [Telefono] <> ""
    )
in
    sin_nulos
```

**Ejemplo avanzado:**

```m
let
    origen = Csv.Document(
        File.Contents("datos.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"Monto", type number}
    ),
    
    // Eliminar filas con error en Monto
    sin_errores = Table.SelectRows(
        cambiar_tipos,
        each not IsError([Monto])
    )
in
    sin_errores
```

---

### 6. Filtrado con múltiples condiciones

```m
let
    origen = Csv.Document(
        File.Contents("solicitudes.csv"),
        [Delimiter=","]
    ),
    con_encabezados = Table.PromoteHeaders(origen),
    cambiar_tipos = Table.TransformColumnTypes(
        con_encabezados,
        {"Monto", type number, "Fecha", type date}
    ),
    
    // Filtrar por múltiples condiciones usando AND
    filtrar_multiples = Table.SelectRows(
        cambiar_tipos,
        each [Estado] = "Pendiente" and
             [Monto] > 500000 and
             [Fecha] >= #date(2024, 1, 1)
    )
in
    filtrar_multiples
```

**Operadores lógicos:**
- `and`: Ambas condiciones deben ser verdaderas
- `or`: Al menos una condición debe ser verdadera
- `not`: Niega la condición

**Ejemplo con OR:**

```m
each [Estado] = "Cancelado" or [Estado] = "Rechazado"
```

---

## Buenas prácticas integradas

### Ejemplo completo: Preparación de datos de solicitudes

```m
// =============================================
// CONSULTA: Preparación de Solicitudes Ciudadanas
// Fecha: 2024-01-15
// Descripción: Limpieza y transformación de solicitudes
// =============================================

let
    // ETAPA 1: CARGA Y LIMPIEZA ESTRUCTURAL
    origen = Excel.Workbook(
        File.Contents("solicitudes_atencion.xlsx"),
        null,
        true
    ),
    tabla = origen{0}[Data],
    con_encabezados = Table.PromoteHeaders(tabla),
    
    // Eliminar columnas innecesarias
    sin_columnas = Table.RemoveColumns(
        con_encabezados,
        {"ColumnaTemporal", "NoUsada"}
    ),
    
    // Eliminar filas vacías
    sin_vacias = Table.SelectRows(
        sin_columnas,
        each not List.IsEmpty(
            List.RemoveMatchingItems(Record.FieldValues(_), {null, ""})
        )
    ),
    
    // ETAPA 2: TRANSFORMACIÓN DE TIPOS
    cambiar_tipos = Table.TransformColumnTypes(
        sin_vacias,
        {
            {"ID_Solicitud", Int64.Type},
            {"Fecha_Solicitud", type date},
            {"Monto_Estimado", type number},
            {"Prioridad", type text},
            {"Comuna", type text},
            {"Estado", type text}
        }
    ),
    
    // ETAPA 3: NORMALIZACIÓN Y REEMPLAZO
    normalizar_texto = Table.TransformColumns(
        cambiar_tipos,
        {
            {"Prioridad", Text.Upper, type text},
            {"Estado", Text.Upper, type text}
        }
    ),
    
    // Reemplazar valores inconsistentes
    reemplazar_prioridad = Table.ReplaceValue(
        normalizar_texto,
        "MEDIA",
        "NORMAL",
        Replacer.ReplaceText,
        {"Prioridad"}
    ),
    
    // ETAPA 4: AGREGAR COLUMNAS CALCULADAS
    agregar_año = Table.AddColumn(
        reemplazar_prioridad,
        "Año_Solicitud",
        each Date.Year([Fecha_Solicitud]),
        type number
    ),
    
    agregar_mes = Table.AddColumn(
        agregar_año,
        "Mes_Solicitud",
        each Date.Month([Fecha_Solicitud]),
        type number
    ),
    
    // ETAPA 5: FILTRADO FINAL
    // Solo solicitudes de 2024 en adelante
    filtrar_fechas = Table.SelectRows(
        agregar_mes,
        each [Año_Solicitud] >= 2024
    ),
    
    // Excluir solicitudes anuladas
    filtrar_estado = Table.SelectRows(
        filtrar_fechas,
        each [Estado] <> "ANULADA"
    ),
    
    // ETAPA 6: REORDENAMIENTO FINAL
    reordenar = Table.ReorderColumns(
        filtrar_estado,
        {
            "ID_Solicitud",
            "Fecha_Solicitud",
            "Año_Solicitud",
            "Mes_Solicitud",
            "Comuna",
            "Prioridad",
            "Estado",
            "Monto_Estimado"
        }
    )
    
in
    reordenar
```

**Estructura recomendada:**
1. Carga y limpieza estructural (columnas, filas vacías)
2. Transformación de tipos
3. Normalización y reemplazo
4. Columnas calculadas
5. Filtrado
6. Reordenamiento final

---

## Referencias rápidas

### Funciones frecuentes de transformación

**Manejo de columnas:**
- `Table.RemoveColumns()`: Elimina columnas
- `Table.RenameColumns()`: Renombra columnas
- `Table.ReorderColumns()`: Reordena columnas
- `Table.SelectColumns()`: Mantiene solo columnas especificadas
- `Table.AddColumn()`: Agrega nueva columna
- `Table.SplitColumn()`: Divide columna
- `Table.CombineColumns()`: Combina columnas

**Manejo de filas:**
- `Table.SelectRows()`: Filtra filas
- `Table.RemoveRowsWithErrors()`: Elimina filas con error
- `Table.Distinct()`: Elimina duplicados
- `Table.FirstN()`: Primeras N filas
- `Table.RemoveFirstN()`: Elimina primeras N filas
- `Table.Sort()`: Ordena filas
- `Table.Reverse()`: Invierte orden

**Agregación:**
- `Table.Group()`: Agrupa por columna
- `Table.Pivot()`: Pivota columna
- `Table.Unpivot()`: Despivota columnas

**Modificación de valores:**
- `Table.ReplaceValue()`: Reemplaza valores
- `Table.TransformColumnTypes()`: Cambia tipo de dato
- `Table.TransformColumns()`: Aplica función a columna

---

## Errores comunes y soluciones

| Error | Causa | Solución |
|-------|-------|----------|
| Columna no encontrada | Nombre mal escrito o sensibilidad a mayúsculas | Verificar exactitud del nombre en [nombre_columna] |
| Error de tipo en operación | Sumar texto o restar fechas incorrectamente | Convertir tipos antes con Table.TransformColumnTypes |
| Duplicados no se eliminan | Comparación case-sensitive | Usar Comparer.OrdinalIgnoreCase |
| Reemplazo no funciona | Espacios ocultos o caracteres invisibles | Aplicar Text.Trim primero |
| Filtro elimina todos los datos | Condición demasiado restrictiva | Revisar lógica y probar con muestra pequeña |
| Agrupación no funciona | Valores nulos en columna de grupo | Filtrar nulos antes de agrupar |
| División de columna incompleta | Delimitador inconsistente | Revisar datos y usar QuoteStyle.Csv si es necesario |

---

## Notas finales

- Siempre verificar el perfil de columna antes y después de transformar
- Documentar cada paso con nombres claros y significativos
- Probar cambios con muestra pequeña antes de aplicar a todo el dataset
- Mantener copia de seguridad de consultas complejas
- Usar parámetros para valores que pueden cambiar
- Separar transformaciones complejas en múltiples pasos legibles
- Considerar rendimiento al filtrar grandes volúmenes (aplicar filtros temprano)